# LIGER default - scib_sim1

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'scib_sim1'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'Batch'
LABEL_KEY = 'Group'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'scib_sim1'
assert BATCH_KEY == 'Batch'
assert LABEL_KEY == 'Group'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                              scib_sim1
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                Batch
label_key                                                Group
metrics      [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=12097 n_batches=6 n_labels=7
ISOLISI preflight: min_batch_coverage=2 isolated_labels=['Group7']
CPCR preflight: eligible_labels=7


In [4]:
LIGER_DEFAULT_ARGS = {
    "k": 20,
    "value_lambda": 5.0,
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_default(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_DEFAULT_ARGS["k"],
        lam=LIGER_DEFAULT_ARGS["value_lambda"],
        max_iterations=LIGER_DEFAULT_ARGS["max_iters"],
        mode=LIGER_DEFAULT_ARGS["mode"],
        layer=LIGER_DEFAULT_ARGS["layer"],
        seed=seed,
    )


pd.Series(LIGER_DEFAULT_ARGS, name="default_setting")


k                   20
value_lambda       5.0
max_iters           30
mode              core
layer           counts
Name: default_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_default(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 28 genes not expressing in Batch1.
Removing 26 genes not expressing in Batch2.


Removing 28 genes not expressing in Batch3.
Removing 24 genes not expressing in Batch4.
Index(['Gene148', 'Gene1364', 'Gene1573', 'Gene1596', 'Gene1989', 'Gene2028',
       'Gene2271', 'Gene2661', 'Gene3065', 'Gene3185', 'Gene3587', 'Gene4753',
       'Gene5428', 'Gene5635', 'Gene6048', 'Gene6098', 'Gene6420', 'Gene7428',
       'Gene8000', 'Gene8427', 'Gene8705', 'Gene8711', 'Gene9481', 'Gene9519'],
      dtype='object', name='gene')


Removing 27 genes not expressing in Batch5.
Removing 27 genes not expressing in Batch6.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7729734389052753


optimised clustering against liger_metric_label
optimal cluster resolution: 0.2
optimal score: 0.9238386808403789
BRAS...


seed           10.000000
BAMI            0.923839
CASW            0.640204
SCGRAPHM        0.429121
ISOLISI         0.981894
BRAS            0.960460
KBET            0.833946
CPCR            0.988342
bio_score       0.743764
batch_score     0.927583
joint_score     0.817292
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.915701,0.659620,0.563042,1.000000,0.858983,0.602514,0.911163,0.784591,0.790886,0.787109
1,2,0.971584,0.688954,0.497149,1.000000,0.918104,0.813948,0.979888,0.789422,0.903980,0.835245
2,3,0.901442,0.637498,0.571281,0.895580,0.897594,0.738227,0.947762,0.751450,0.861194,0.795348
3,4,0.835501,0.659449,0.405611,0.970705,0.840752,0.633219,0.959466,0.717816,0.811146,0.755148
4,5,0.956636,0.666879,0.487641,1.000000,0.935850,0.759588,0.970469,0.777789,0.888636,0.822128
5,6,0.950169,0.680305,0.650456,1.000000,0.869141,0.711416,0.948206,0.820232,0.842921,0.829308
6,7,0.986332,0.683189,0.595987,1.000000,0.946992,0.800236,0.987306,0.816377,0.911511,0.854431
7,8,0.932636,0.664475,0.507258,1.000000,0.876458,0.700825,0.932125,0.776092,0.836469,0.800243
8,9,0.957340,0.690844,0.512678,1.000000,0.919457,0.811886,0.974354,0.790215,0.901899,0.834889
9,10,0.923839,0.640204,0.429121,0.981894,0.960460,0.833946,0.988342,0.743764,0.927583,0.817292


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.933118,0.040821,0.835501,0.986332
CASW,0.667142,0.017876,0.637498,0.690844
SCGRAPHM,0.522022,0.070967,0.405611,0.650456
ISOLISI,0.984818,0.031280,0.895580,1.000000
BRAS,0.902379,0.038084,0.840752,0.960460
KBET,0.740581,0.074920,0.602514,0.833946
CPCR,0.959908,0.023898,0.911163,0.988342
bio_score,0.776775,0.030112,0.717816,0.820232
batch_score,0.867623,0.043772,0.790886,0.927583
joint_score,0.813114,0.027476,0.755148,0.854431
